# Introduction to MediaDir

This notebook demonstrates the MediaDir functionality from `mediatools`, which provides a powerful recursive data structure for organizing and managing media files in filesystem directories.

**What you'll learn:**
- How to scan directories for media files (videos, images, other files)
- Navigate and work with nested directory structures
- Access and process media files at individual and batch levels
- Compare directories and detect changes
- Integration patterns with other mediatools modules

**Prerequisites:**
- Basic Python knowledge
- Familiarity with pathlib for file system operations
- Some media files to experiment with (we'll create mock examples)

**Key Concepts:**
- **MediaDir**: Recursive directory structure containing videos, images, and subdirectories
- **File Collections**: VideoFiles, ImageFiles, and other file containers
- **Metadata Integration**: Each file type provides rich metadata access
- **Path Flexibility**: Support for both absolute and relative path handling

## Import Section

In [1]:
import pathlib
import tempfile
import shutil
from pathlib import Path
import mediatools

## Setup and Mock Data Creation

Since this is a demonstration notebook, we'll create a realistic directory structure with mock media files. This allows you to run the examples without needing your own media collection.

In [2]:
# Create a temporary directory structure with mock media files
def create_mock_media_directory():
    """Create a realistic directory structure with mock media files for demonstration"""
    temp_dir = Path(tempfile.mkdtemp(prefix="mediatools_demo_"))
    print(f"Created demo directory: {temp_dir}")
    
    # Define all files to create (directories will be created automatically)
    mock_files = [
        # Videos
        "vacation_2023/beach/surfing.mp4", "vacation_2023/beach/sunset.mov",
        "vacation_2023/mountains/hiking.mp4", "vacation_2023/mountains/drone_footage.mp4", 
        "family_events/birthday/cake_ceremony.mp4", "family_events/birthday/kids_playing.avi",
        "projects/timelapse/construction.mkv",
        # Images  
        "vacation_2023/beach/group_photo.jpg", "vacation_2023/beach/shells.png",
        "vacation_2023/mountains/panorama.jpg", "vacation_2023/mountains/wildlife.jpg",
        "family_events/birthday/decorations.jpg", "family_events/family_portrait.png",
        "projects/timelapse/setup.jpg",
        # Other files
        "vacation_2023/itinerary.txt", "family_events/guest_list.xlsx", "projects/README.md"
    ]
    
    # Create all files (directories will be created automatically)
    for file_path in mock_files:
        full_path = temp_dir / file_path
        full_path.parent.mkdir(parents=True, exist_ok=True)
        full_path.touch()
    
    return temp_dir

# Create our demo directory
demo_root = create_mock_media_directory()

print("\nDemo directory structure created:")
for item in sorted(demo_root.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(demo_root)}")

Created demo directory: /tmp/mediatools_demo_3z61ly7i

Demo directory structure created:
  family_events/birthday/cake_ceremony.mp4
  family_events/birthday/decorations.jpg
  family_events/birthday/kids_playing.avi
  family_events/family_portrait.png
  family_events/guest_list.xlsx
  projects/README.md
  projects/timelapse/construction.mkv
  projects/timelapse/setup.jpg
  vacation_2023/beach/group_photo.jpg
  vacation_2023/beach/shells.png
  vacation_2023/beach/sunset.mov
  vacation_2023/beach/surfing.mp4
  vacation_2023/itinerary.txt
  vacation_2023/mountains/drone_footage.mp4
  vacation_2023/mountains/hiking.mp4
  vacation_2023/mountains/panorama.jpg
  vacation_2023/mountains/wildlife.jpg


## Creating and Navigating `MediaDir` Instances

### Creating a MediaDir from a Directory

The primary way to create a MediaDir is by scanning a filesystem directory. MediaDir will automatically categorize files into videos, images, and other files based on their extensions.

In [3]:
# Method 1: Using the scan_directory function (recommended)
media_dir = mediatools.scan_directory(demo_root)

print(f"MediaDir created for: {media_dir.path}")
print(f"Contains {len(media_dir.subdirs)} subdirectories, {len(media_dir.videos)} videos, {len(media_dir.images)} images, and {len(media_dir.other_files)} other files in total")

MediaDir created for: /tmp/mediatools_demo_3z61ly7i
Contains 3 subdirectories, 0 videos, 0 images, and 0 other files in total


In [4]:
# Method 2: Using the MediaDir class method directly
media_dir_alt = mediatools.MediaDir.from_path(demo_root)

# Both methods produce equivalent results
print(f"Same result: {media_dir.path == media_dir_alt.path}")

Same result: True


You can use the `display` method to see an overview of the directory structure.

In [5]:
print(media_dir.display())

/tmp/mediatools_demo_3z61ly7i/
├── family_events/
│   ├── [I] family_portrait.png
│   ├── [F] guest_list.xlsx
│   └── birthday/
│       ├── [V] cake_ceremony.mp4
│       ├── [V] kids_playing.avi
│       └── [I] decorations.jpg
├── projects/
│   ├── [F] README.md
│   └── timelapse/
│       ├── [V] construction.mkv
│       └── [I] setup.jpg
└── vacation_2023/
    ├── [F] itinerary.txt
    ├── beach/
    │   ├── [V] sunset.mov
    │   ├── [V] surfing.mp4
    │   ├── [I] group_photo.jpg
    │   └── [I] shells.png
    └── mountains/
        ├── [V] drone_footage.mp4
        ├── [V] hiking.mp4
        ├── [I] panorama.jpg
        └── [I] wildlife.jpg


Either method accepts `video_ext` and `image_ext` parameters to control which file types are considered as video or images. From the `display` output see that the .mkv files are no longer seen as files (they show `F` instead of `V`).

In [24]:
md = mediatools.scan_directory(
    demo_root,
    video_ext=['.mp4', '.mov'],  # Only MP4 and MOV files
    image_ext=['.jpg', '.jpeg']  # Only JPEG files
)
print(md.display())

/tmp/mediatools_demo_3z61ly7i/
├── family_events/
│   ├── [F] family_portrait.png
│   ├── [F] guest_list.xlsx
│   └── birthday/
│       ├── [V] cake_ceremony.mp4
│       ├── [I] decorations.jpg
│       └── [F] kids_playing.avi
├── projects/
│   ├── [F] README.md
│   └── timelapse/
│       ├── [I] setup.jpg
│       └── [F] construction.mkv
└── vacation_2023/
    ├── [F] itinerary.txt
    ├── beach/
    │   ├── [V] sunset.mov
    │   ├── [V] surfing.mp4
    │   ├── [I] group_photo.jpg
    │   └── [F] shells.png
    └── mountains/
        ├── [V] drone_footage.mp4
        ├── [V] hiking.mp4
        ├── [I] panorama.jpg
        └── [I] wildlife.jpg


The `ignore_path` allows you to ignore particular paths. Note that this is applied recursively, so any matching path will be ignored as well as all of its subdirectories.

Here I ignore all directories that start with "b".

In [39]:
md = mediatools.scan_directory(
    demo_root,
    ignore_path = lambda p: p.name.startswith('b'),
)
print(md.display())

/tmp/mediatools_demo_3z61ly7i/
├── family_events/
│   ├── [I] family_portrait.png
│   └── [F] guest_list.xlsx
├── projects/
│   ├── [F] README.md
│   └── timelapse/
│       ├── [V] construction.mkv
│       └── [I] setup.jpg
└── vacation_2023/
    ├── [F] itinerary.txt
    └── mountains/
        ├── [V] drone_footage.mp4
        ├── [V] hiking.mp4
        ├── [I] panorama.jpg
        └── [I] wildlife.jpg


### Navigating Directory Trees

Directories are trees, so you can navigate them as such.

Access subdirectories using the `subdirs` attribute. This is a dictionary of subdir names mapped to `MediaDir` instances.

In [7]:
print(f"{media_dir} has {len(media_dir.subdirs)} subdirectories:")
for subdir_name, subdir in media_dir.subdirs.items():
    print(f"  {subdir_name}: {subdir.path}")

MediaDir("/tmp/mediatools_demo_3z61ly7i") has 3 subdirectories:
  family_events: /tmp/mediatools_demo_3z61ly7i/family_events
  vacation_2023: /tmp/mediatools_demo_3z61ly7i/vacation_2023
  projects: /tmp/mediatools_demo_3z61ly7i/projects


Each `MediaDir` also has a parent.

In [9]:
for subdir in media_dir.subdirs.values():
    print(subdir.parent)

MediaDir("/tmp/mediatools_demo_3z61ly7i")
MediaDir("/tmp/mediatools_demo_3z61ly7i")
MediaDir("/tmp/mediatools_demo_3z61ly7i")


Note that the root instance parent is `None`.

In [8]:
print(media_dir.parent)

None


The fact that this is a recursive data structure means it is easy to write recursive functions.

In [30]:
def print_tree(mdir: mediatools.MediaDir, level: int = 0):
    print(f"{'  ' * level}- {mdir.path.name}")
    for subdir in mdir.subdirs.values():
        print_tree(subdir, level + 1)

print_tree(media_dir)

- mediatools_demo_3z61ly7i
  - family_events
    - birthday
  - vacation_2023
    - mountains
    - beach
  - projects
    - timelapse


In [32]:
def count_files(mdir: mediatools.MediaDir) -> int:
    count = len(mdir.videos) + len(mdir.images) + len(mdir.other_files)
    for subdir in mdir.subdirs.values():
        count += count_files(subdir)
    return count
count_files(media_dir)

17

### Media Directories as Paths

There are also more path-centric methods for interacting with the media directories.

You can use subscripts to access child directories. Note that you cannot use this to access files - only subdirectories.

In [11]:
media_dir["vacation_2023"]["mountains"]

MediaDir("/tmp/mediatools_demo_3z61ly7i/vacation_2023/mountains")

You can also use the `subdir` method to access children. Note that it accepts a variable number of arguments.

In [12]:
media_dir.subdir("vacation_2023"), media_dir.subdir("vacation_2023", "beach")

(MediaDir("/tmp/mediatools_demo_3z61ly7i/vacation_2023"),
 MediaDir("/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach"))

The `subdir` method also accepts full relative paths.

In [13]:
media_dir.subdir(Path("family_events") / "birthday")

MediaDir("/tmp/mediatools_demo_3z61ly7i/family_events/birthday")

## Working with Media Files

`MediaDir` instances automatically track videos and images in `VideoFile` and `ImageFile` instances, and all other files are wrapped in `NonMediaFile` instances. You can access videos through `.videos`, images through `.images`, and other files through `.other_files`.

This is a set of methods and properties you can use to access files from a `MediaDir`:

### Properties for Current Directory
- `videos` (`list[VideoFile]`): videos in the represented directory only.
- `images` (`list[ImageFile]`): images in the represented directory only.
- `other_files` (`list[NonMediaFile]`): all non-media files in the directory only.

### Path Methods for Current Directory
- `video_paths()` (`list[Path]`): list of video file paths in the directory only.
- `image_paths()` (`list[Path]`): list of image file paths in the directory only.

### Recursive Methods (Include Subdirectories)
- `all_video_files()` (`list[VideoFile]`): all video files recursively across directory tree.
- `all_image_files()` (`list[ImageFile]`): all image files recursively across directory tree.
- `all_video_paths()` (`list[Path]`): all video file paths recursively across directory tree.
- `all_image_paths()` (`list[Path]`): all image file paths recursively across directory tree.
- `all_file_paths()` (`list[Path]`): all file paths (including non-media) recursively across directory tree.
- `all_media_paths()` (`list[Path]`): all media file paths (videos + images) recursively across directory tree.

### Get Specific Files by Path
- `get_video(path)` (`VideoFile`): retrieve a specific video file by its path.
- `get_image(path)` (`ImageFile`): retrieve a specific image file by its path.
- `get_nonmedia(path)` (`NonMediaFile`): retrieve a specific non-media file by its path.

In [60]:
beach_dir = media_dir["vacation_2023"]["beach"]
beach_dir.videos

[VideoFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/surfing.mp4'), meta={}),
 VideoFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/sunset.mov'), meta={})]

In [61]:
beach_dir.images

[ImageFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/group_photo.jpg'), meta={}),
 ImageFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/shells.png'), meta={})]

You can also retrieve all video or image files recursively from a media directory using `all_video_files` or `all_image_files` methods.

In [57]:
media_dir.all_video_files()

[VideoFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/family_events/birthday/cake_ceremony.mp4'), meta={}),
 VideoFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/family_events/birthday/kids_playing.avi'), meta={}),
 VideoFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/mountains/drone_footage.mp4'), meta={}),
 VideoFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/mountains/hiking.mp4'), meta={}),
 VideoFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/surfing.mp4'), meta={}),
 VideoFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/sunset.mov'), meta={}),
 VideoFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/projects/timelapse/construction.mkv'), meta={})]

In [58]:
media_dir.all_image_files()

[ImageFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/family_events/family_portrait.png'), meta={}),
 ImageFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/family_events/birthday/decorations.jpg'), meta={}),
 ImageFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/mountains/wildlife.jpg'), meta={}),
 ImageFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/mountains/panorama.jpg'), meta={}),
 ImageFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/group_photo.jpg'), meta={}),
 ImageFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/shells.png'), meta={}),
 ImageFile(path=PosixPath('/tmp/mediatools_demo_3z61ly7i/projects/timelapse/setup.jpg'), meta={})]

In [62]:
media_dir.all_video_paths()

[PosixPath('/tmp/mediatools_demo_3z61ly7i/family_events/birthday/cake_ceremony.mp4'),
 PosixPath('/tmp/mediatools_demo_3z61ly7i/family_events/birthday/kids_playing.avi'),
 PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/mountains/drone_footage.mp4'),
 PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/mountains/hiking.mp4'),
 PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/surfing.mp4'),
 PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/sunset.mov'),
 PosixPath('/tmp/mediatools_demo_3z61ly7i/projects/timelapse/construction.mkv')]

In [63]:
media_dir.image_paths()

[]

All file instance types have a `stat` method that returns a `FileStatResult` type, which is essentially a `pydantic.BaseType` containing the same information as `os.stat_result` with some convenient methods.

In [50]:
ex_vid = beach_dir.videos[0]
ex_vid.path

PosixPath('/tmp/mediatools_demo_3z61ly7i/vacation_2023/beach/surfing.mp4')

In [17]:
stat = ex_vid.stat()
stat.size_str(), stat.modified_at_str(), stat.accessed_at_str(), stat.changed_at_str()

('0.00 Bytes',
 '2026-04-25 18:26:57 UTC',
 '2026-04-25 18:26:57 UTC',
 '2026-04-25 18:26:57 UTC')

## Recursive File Access

### Getting All Files Across the Entire Directory Tree

One of MediaDir's most powerful features is the ability to collect all files recursively across the entire directory structure.

In [ ]:
# Get all video files recursively
all_videos = media_dir.all_video_files()
print(f"Total videos found: {len(all_videos)}")
print("All video files:")
for video in all_videos:
    rel_path = video.path.relative_to(demo_root)
    print(f"  {rel_path}")

print(f"\nTotal images found: {len(media_dir.all_image_files())}")
print("All image files:")
for image in media_dir.all_image_files():
    rel_path = image.path.relative_to(demo_root)
    print(f"  {rel_path}")

Total videos found: 7
All video files:
  family_events/birthday/cake_ceremony.mp4
  family_events/birthday/kids_playing.avi
  vacation_2023/mountains/drone_footage.mp4
  vacation_2023/mountains/hiking.mp4
  vacation_2023/beach/surfing.mp4
  vacation_2023/beach/sunset.mov
  projects/timelapse/construction.mkv

Total images found: 7
All image files:
  family_events/family_portrait.png
  family_events/birthday/decorations.jpg
  vacation_2023/mountains/wildlife.jpg
  vacation_2023/mountains/panorama.jpg
  vacation_2023/beach/group_photo.jpg
  vacation_2023/beach/shells.png
  projects/timelapse/setup.jpg


In [ ]:
# Get all file paths (including non-media files)
all_file_paths = media_dir.all_file_paths()
media_file_paths = media_dir.all_media_paths()

print(f"All files: {len(all_file_paths)}")
print(f"Media files only: {len(media_file_paths)}")
print(f"Non-media files: {len(all_file_paths) - len(media_file_paths)}")

All files: 14
Media files only: 14
Non-media files: 0


## Advanced File Retrieval

### Finding Specific Files by Path

In [ ]:
# Define paths to specific files
video_path = demo_root / "vacation_2023" / "beach" / "surfing.mp4"
image_path = demo_root / "vacation_2023" / "mountains" / "panorama.jpg"
other_path = demo_root / "vacation_2023" / "itinerary.txt"

Found video: /tmp/mediatools_demo_yt_qbtdz/vacation_2023/beach/surfing.mp4
Found image: /tmp/mediatools_demo_yt_qbtdz/vacation_2023/mountains/panorama.jpg
Found other file: /tmp/mediatools_demo_yt_qbtdz/vacation_2023/itinerary.txt


In [ ]:
# Find specific files by path
video_file = media_dir.get_video(video_path)
video_file

In [ ]:
image_file = media_dir.get_image(image_path)
image_file

In [ ]:
other_file = media_dir.get_nonmedia(other_path)
other_file

## Configuration Options

### Custom File Extensions and Filtering

In [ ]:
# Create MediaDir with custom file extensions
custom_media_dir = mediatools.scan_directory(
    demo_root,
    video_ext=['.mp4', '.mov'],  # Only MP4 and MOV files
    image_ext=['.jpg', '.jpeg']  # Only JPEG files
)

With custom extensions:
Videos found: 5
Images found: 5

With default extensions:
Videos found: 7
Images found: 7


In [ ]:
# Compare custom vs default extensions
len(custom_media_dir.all_video_files()), len(custom_media_dir.all_image_files())

In [ ]:
len(media_dir.all_video_files()), len(media_dir.all_image_files())

In [ ]:
# Define path filtering function
def ignore_timelapse_projects(path):
    """Ignore any paths containing 'timelapse' or 'projects'"""
    return 'timelapse' in str(path) or 'projects' in str(path)

With path filtering (ignoring timelapse/projects):
Videos found: 6
Subdirectories: ['family_events', 'vacation_2023']


In [ ]:
# Create filtered MediaDir
filtered_media_dir = mediatools.scan_directory(
    demo_root,
    ignore_path=ignore_timelapse_projects
)

In [ ]:
# Show filtering results
len(filtered_media_dir.all_video_files()), list(filtered_media_dir.subdirs.keys())

In [ ]:
# Create MediaDir with relative paths
relative_media_dir = mediatools.scan_directory(
    demo_root,
    use_absolute=False
)

Path handling comparison:
Absolute path: /tmp/mediatools_demo_yt_qbtdz
Relative path: .


In [ ]:
# Compare path handling
media_dir.path, relative_media_dir.path

## Directory Comparison and Synchronization

### Detecting Changes Between Directory States

In [ ]:
# Create a modified version of our directory for comparison
def create_modified_directory():
    """Create a slightly modified version of our demo directory"""
    modified_root = Path(tempfile.mkdtemp(prefix="mediatools_modified_"))
    
    # Copy the original structure
    shutil.copytree(demo_root, modified_root, dirs_exist_ok=True)
    
    # Make some changes:
    # 1. Remove a file
    (modified_root / "vacation_2023" / "beach" / "shells.png").unlink()
    
    # 2. Add a new file
    (modified_root / "vacation_2023" / "new_video.mp4").touch()
    
    # 3. Add a new directory with files
    new_dir = modified_root / "vacation_2023" / "city"
    new_dir.mkdir()
    (new_dir / "architecture.jpg").touch()
    (new_dir / "street_performance.mp4").touch()
    
    return modified_root

Original directory files: 14
Modified directory files: 16


In [ ]:
# Create modified directory and scan it
modified_root = create_modified_directory()
modified_media_dir = mediatools.scan_directory(modified_root)

In [ ]:
# Compare file counts
len(media_dir.all_file_paths()), len(modified_media_dir.all_file_paths())

In [ ]:
# Compare the two directory structures
removed_files, added_files = media_dir.file_diff(modified_media_dir)

Files removed in modified version:
  - /tmp/mediatools_demo_yt_qbtdz/vacation_2023/beach/shells.png
  - /tmp/mediatools_demo_yt_qbtdz/vacation_2023/beach/surfing.mp4
  - /tmp/mediatools_demo_yt_qbtdz/vacation_2023/mountains/drone_footage.mp4
  - /tmp/mediatools_demo_yt_qbtdz/vacation_2023/mountains/hiking.mp4
  - /tmp/mediatools_demo_yt_qbtdz/projects/timelapse/construction.mkv
  - /tmp/mediatools_demo_yt_qbtdz/family_events/family_portrait.png
  - /tmp/mediatools_demo_yt_qbtdz/vacation_2023/mountains/panorama.jpg
  - /tmp/mediatools_demo_yt_qbtdz/family_events/birthday/cake_ceremony.mp4
  - /tmp/mediatools_demo_yt_qbtdz/family_events/birthday/kids_playing.avi
  - /tmp/mediatools_demo_yt_qbtdz/vacation_2023/mountains/wildlife.jpg
  - /tmp/mediatools_demo_yt_qbtdz/family_events/birthday/decorations.jpg
  - /tmp/mediatools_demo_yt_qbtdz/vacation_2023/beach/group_photo.jpg
  - /tmp/mediatools_demo_yt_qbtdz/vacation_2023/beach/sunset.mov
  - /tmp/mediatools_demo_yt_qbtdz/projects/timelapse

In [ ]:
# Files removed in modified version
removed_files

In [ ]:
# Files added in modified version
added_files

In [ ]:
# Find directories that have changes
changed_dirs = media_dir.get_changed_dirs(modified_media_dir)

Directories with changes:


In [ ]:
# Show directories with changes
[changed_dir.path.relative_to(demo_root) for changed_dir in changed_dirs]

## Integration with Other MediaTools Modules

### Working with VideoFile and ImageFile Objects

In [ ]:
# Access individual video file objects
video_files = media_dir.all_video_files()[:2]  # First 2 videos

Video file integration examples:

cake_ceremony.mp4:
  Type: VideoFile
  Path: /tmp/mediatools_demo_yt_qbtdz/family_events/birthday/cake_ceremony.mp4
  Available methods: probe(), read_meta(), ffmpeg(), etc.

kids_playing.avi:
  Type: VideoFile
  Path: /tmp/mediatools_demo_yt_qbtdz/family_events/birthday/kids_playing.avi
  Available methods: probe(), read_meta(), ffmpeg(), etc.

Image file integration examples:

family_portrait.png:
  Type: ImageFile
  Path: /tmp/mediatools_demo_yt_qbtdz/family_events/family_portrait.png
  Available methods: read(), read_meta(), etc.

decorations.jpg:
  Type: ImageFile
  Path: /tmp/mediatools_demo_yt_qbtdz/family_events/birthday/decorations.jpg
  Available methods: read(), read_meta(), etc.


In [ ]:
# Show video file types and paths
[(type(vf).__name__, vf.path.name) for vf in video_files]

In [ ]:
# Access individual image file objects
image_files = media_dir.all_image_files()[:2]  # First 2 images

In [ ]:
# Show image file types and paths
[(type(if_).__name__, if_.path.name) for if_ in image_files]

## Serialization and Data Export

### Converting MediaDir to Dictionary Formats

In [ ]:
# Convert to dictionary representation
media_dict = media_dir.to_dict()

MediaDir dictionary representation (top level):
  path: /tmp/mediatools_demo_yt_qbtdz
  videos: [0 items]
  images: [0 items]
  other_files: [0 items]
  subdirs: [{'path': '/tmp/mediatools_demo_yt_qbtdz/family_events', 'videos': [], 'images': [{'path': '/tmp/mediatools_demo_yt_qbtdz/family_events/family_portrait.png', 'meta': {}}], 'other_files': [{'path': '/tmp/mediatools_demo_yt_qbtdz/family_events/guest_list.xlsx', 'meta': {}}], 'subdirs': [{'path': '/tmp/mediatools_demo_yt_qbtdz/family_events/birthday', 'videos': [{'path': '/tmp/mediatools_demo_yt_qbtdz/family_events/birthday/cake_ceremony.mp4', 'meta': {}}, {'path': '/tmp/mediatools_demo_yt_qbtdz/family_events/birthday/kids_playing.avi', 'meta': {}}], 'images': [{'path': '/tmp/mediatools_demo_yt_qbtdz/family_events/birthday/decorations.jpg', 'meta': {}}], 'other_files': [], 'subdirs': [], 'meta': {}}], 'meta': {}}, {'path': '/tmp/mediatools_demo_yt_qbtdz/vacation_2023', 'videos': [], 'images': [], 'other_files': [{'path': '/tmp/me

In [ ]:
# Show top-level dictionary structure
{k: (list(v.keys()) if k == 'subdirs' and isinstance(v, dict) 
     else len(v) if isinstance(v, list) 
     else v) for k, v in media_dict.items()}

## Practical Use Cases and Workflows

### 1. Media Library Organization

In [ ]:
def analyze_media_library(media_dir):
    """Generate a comprehensive analysis of a media library"""
    
    analysis = {
        'total_directories': len(list(media_dir.path.rglob('*'))) if media_dir.path.exists() else 0,
        'total_videos': len(media_dir.all_video_files()),
        'total_images': len(media_dir.all_image_files()),
        'total_other_files': len(media_dir.all_file_paths()) - len(media_dir.all_media_paths()),
        'directory_breakdown': {}
    }
    
    # Analyze each subdirectory
    for subdir_name, subdir in media_dir.subdirs.items():
        analysis['directory_breakdown'][subdir_name] = {
            'videos': len(subdir.all_video_files()),
            'images': len(subdir.all_image_files()),
            'subdirectories': len(subdir.subdirs)
        }
    
    return analysis

In [ ]:
# Analyze our demo library
analysis = analyze_media_library(media_dir)

In [ ]:
# Show total counts
analysis['total_videos'], analysis['total_images'], analysis['total_other_files']

In [ ]:
# Show directory breakdown
analysis['directory_breakdown']

### 2. Batch Processing Preparation

In [ ]:
def prepare_batch_processing_list(media_dir, file_type='video'):
    """Prepare a list of files for batch processing with organized metadata"""
    
    if file_type == 'video':
        files = media_dir.all_video_files()
    elif file_type == 'image':
        files = media_dir.all_image_files()
    else:
        raise ValueError("file_type must be 'video' or 'image'")
    
    processing_list = []
    for file_obj in files:
        # Get relative path for organization
        rel_path = file_obj.path.relative_to(media_dir.path)
        
        # Determine category based on directory structure
        path_parts = rel_path.parts
        category = path_parts[0] if len(path_parts) > 1 else 'root'
        subcategory = path_parts[1] if len(path_parts) > 2 else None
        
        processing_list.append({
            'file_object': file_obj,
            'full_path': file_obj.path,
            'relative_path': rel_path,
            'category': category,
            'subcategory': subcategory,
            'filename': file_obj.path.name,
            'extension': file_obj.path.suffix.lower()
        })
    
    return processing_list

In [ ]:
# Prepare video processing list
video_batch = prepare_batch_processing_list(media_dir, 'video')

In [ ]:
# Show batch structure (first few items)
[(item['category'], item['subcategory'], item['filename']) for item in video_batch[:3]]

In [ ]:
# Group by category for organized processing
by_category = {}
for item in video_batch:
    category = item['category']
    if category not in by_category:
        by_category[category] = []
    by_category[category].append(item)

In [ ]:
# Show files grouped by category
{category: len(items) for category, items in by_category.items()}

### 3. Change Detection and Backup Verification

In [ ]:
def create_backup_verification_report(source_dir, backup_dir):
    """Create a detailed report comparing source and backup directories"""
    
    removed, added = source_dir.file_diff(backup_dir)
    
    report = {
        'backup_complete': len(removed) == 0 and len(added) == 0,
        'missing_from_backup': removed,
        'extra_in_backup': added,
        'source_stats': {
            'total_files': len(source_dir.all_file_paths()),
            'videos': len(source_dir.all_video_files()),
            'images': len(source_dir.all_image_files())
        },
        'backup_stats': {
            'total_files': len(backup_dir.all_file_paths()),
            'videos': len(backup_dir.all_video_files()),
            'images': len(backup_dir.all_image_files())
        }
    }
    
    return report

In [ ]:
# Compare original with modified directory (simulating backup check)
verification = create_backup_verification_report(media_dir, modified_media_dir)

In [ ]:
# Show backup verification results
verification['backup_complete'], verification['source_stats'], verification['backup_stats']

In [ ]:
# Show missing files (first few)
list(verification['missing_from_backup'])[:3]

In [ ]:
# Show extra files (first few)
list(verification['extra_in_backup'])[:3]

## Cleanup

Let's clean up our temporary directories.

In [ ]:
# Clean up temporary directories
try:
    shutil.rmtree(demo_root)
    shutil.rmtree(modified_root)
    "Temporary directories cleaned up successfully."
except Exception as e:
    f"Cleanup warning: {e}"

## Summary

This notebook demonstrated the core functionality of the MediaDir system:

**Key Features Covered:**
- **Directory Scanning**: Automatically categorize media files by type
- **Recursive Navigation**: Easy access to nested directory structures
- **File Collections**: Organized access to videos, images, and other files
- **Batch Operations**: Process all files across directory trees
- **Change Detection**: Compare directory states for synchronization
- **Flexible Configuration**: Custom extensions and filtering options

**Integration Points:**
- VideoFile objects provide FFmpeg integration for video processing
- ImageFile objects enable image manipulation and analysis
- Serialization support for persistence and data export
- Path flexibility for both development and production scenarios

**Common Use Cases:**
1. **Media Library Management**: Organize and catalog large collections
2. **Batch Processing**: Process files systematically across directory trees
3. **Backup Verification**: Ensure completeness of media backups
4. **Change Monitoring**: Detect additions, removals, and modifications
5. **Website Generation**: Build galleries from filesystem structure

MediaDir serves as the foundational data structure that makes complex media workflows simple and intuitive, providing the organizational backbone for the entire mediatools ecosystem.